# P3 — Task 3 — Engagement + soát số toàn deck

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task3_engagement.ipynb)

**Slide sở hữu:** 11, 12, 13 · **Slide mới:** 0 (làm chart tốt hơn) + soát số  
**Đã có:** task3.py, task3_findings.md, chart t3_engagement_dist / t3_channel_mix / t3_diversity_vs_engagement / t3_health_vs_engagement_quadrant

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, sys, gzip, shutil
DRIVE_DATA = "/content/drive/MyDrive/BI10_ROUND01_DATASET"   # thư mục data trên Drive (xem sheet 'Setup Colab')
REPO = "/content/BI10GroupYAPPERS"

from google.colab import drive, userdata
drive.mount("/content/drive")

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
for f in ["consumer_financial_health_engagement_2025.csv", "data_dictionary.xlsx"]:
    if not os.path.exists(f"{D}/{f}") and os.path.exists(f"{DRIVE_DATA}/{f}"):
        shutil.copy(f"{DRIVE_DATA}/{f}", f"{D}/{f}")
if not os.path.exists(TX):
    gz, raw = f"{DRIVE_DATA}/consumer_transactions_2025.parquet.gz", f"{DRIVE_DATA}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
    if os.path.exists(gz):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
        with gzip.open(gz) as src, open(TX, "wb") as dst: shutil.copyfileobj(src, dst)
    else:                    # hoặc upload nguyên thư mục .parquet/ chứa CSV 597MB
        shutil.copy(raw, TX)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    os.system("python draft/task4.py > /dev/null")
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task3_output.txt`.

In [ ]:
%run -i draft/task3.py

## Việc 1 — Deliverable 1 — % khách từng engagement segment  `[BẮT BUỘC · hạn 27/09 12:00]`
Không chỉ '99% high/very high': ghi % từng nhóm lên histogram (nhãn trên cột).

**Giao:** Chart thay thế slide 11

In [ ]:
# Gợi ý:
# cons.engagement_segment.value_counts(normalize=True)


## Việc 2 — Deliverable 2 — average online spend share + ai dùng QR/Mobile  `[Cao · hạn 27/09 12:00]`
Nêu rõ TB online_spend_ratio cấp khách trên slide 11. Thêm kênh theo nhóm tuổi hoặc theo segment của P4 (nhãn trong draft/task4_features.csv).

**Giao:** Bullet slide 11 + gửi P5 (Reminders/Nudges)

In [ ]:
# Gợi ý:
# seg=pd.read_csv('draft/task4_features.csv'); pd.crosstab(t.segment, t.transaction_channel, normalize='index')


## Việc 3 — Deliverable 3 — histogram diversity + kiểm r = +0.94  `[Trung bình · hạn 27/09 18:00]`
Histogram category diversity (2–14) cạnh scatter. Tính lại r chỉ trên khách diversity ≥ 8 để xem có bị đuôi nhỏ kéo không.

**Giao:** Chart + bullet slide 12

In [ ]:
# Gợi ý:
# c=cons[cons.category_diversity>=8]; c.category_diversity.corr(c.engagement)


## Việc 4 — Deliverable 4 — heatmap Recency × Frequency  `[Cao · hạn 27/09 18:00]`
Bin recency × bin active_transaction_days, màu = engagement TB. 'Phù hợp dữ liệu' hơn bảng số (Objective 6).

**Giao:** Chart slide 12

In [ ]:
# Gợi ý:
# mon.pivot_table('engagement_score', pd.cut(mon.transaction_recency_days,5), pd.cut(mon.active_transaction_days,5))


## Việc 5 — Deliverable 5 — bảng độ nhạy cutoff Healthy-but-Disengaged  `[BẮT BUỘC · hạn 27/09 18:00]`
Số khách khi FHS ≥ 65/70/75 × engagement < 60/65/70/75. Đề hỏi 'vì sao chọn cutoff này thay vì chặt/lỏng hơn' → bảng này là câu trả lời.

**Giao:** Bảng slide 13

In [ ]:
# Gợi ý:
# pd.DataFrame({e:[((cons.fhs>=f)&(cons.engagement<e)).sum() for f in (65,70,75)] for e in (60,65,70,75)}, index=[65,70,75])


## Việc 6 — 1 dòng 'vì sao chọn loại chart này' cho mỗi chart  `[Trung bình · hạn 27/09 18:00]`
Ghi trong phần Findings của notebook → P5 đưa vào speaker note.

**Giao:** Speaker note

In [ ]:
# Gợi ý:
# —


## Việc 7 — SOÁT SỐ TOÀN DECK (việc thêm để cân tải)  `[Cao · hạn 28/09 12:00]`
Khi P5 gửi deck v2 (27/09 23:00): so MỌI con số trên slide 2, 16, 17, 18 với output các task (chạy lại %run từng script nếu cần). Lập danh sách lệch → gửi P5 trước 28/09 12:00.

**Giao:** Danh sách lệch số → P5

In [ ]:
# Gợi ý:
# —


## Findings mới (2–4 dòng / việc, có số)
- ...

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task3_engagement.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.